# Does the comparison depend on preprocessing choices?

A walkthrough of the three sensitivity analyses in `scripts/`: the below-detection fill rule
(`run_fill_sensitivity.py`), the scaler (`run_scaling_sensitivity.py`), and whether the fill is
estimated once upfront or separately inside every training fold
(`run_trainonly_sensitivity.py`). Each rebuilds the input matrix (or the preprocessing step)
differently and re-evaluates `sex` at 6 months, raw vs PCA, to see whether the raw-vs-embedding
gap changes. The data in `data/` is purely synthetic, drawn from chosen statistical
distributions, not derived from any real data (see `00_build_data.ipynb`).


In [1]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().parent
SRC_DIR = REPO_ROOT / "src"
DATA_DIR = REPO_ROOT / "data"
sys.path.insert(0, str(SRC_DIR))
sys.path.insert(0, str(SRC_DIR / "analysis"))
# So a subprocess started from here (python "scripts/some_script.py") can "from metabo
# import ..." too, exactly as if the package had been pip installed.
os.environ["PYTHONPATH"] = str(SRC_DIR)
os.environ["METABO_DATA_DIR"] = str(DATA_DIR)
os.environ["METABO_RESULTS_DIR"] = str(Path.cwd() / "results")

# data/ is not shipped with the repo -- generate it on first use, same recipe as
# 00_build_data.ipynb, if it is not already there.
_RENAMED = {
    "datacorr_6m6y18mMw1w24_final.csv": "synthetic_raw_metabolome.csv",
    "target_for_metabolites_ml.csv": "synthetic_targets.csv",
    "metabolites_6m_for_ml.csv": "synthetic_metabolites_6m.csv",
    "metabolites_18m_for_ml.csv": "synthetic_metabolites_18m.csv",
    "metabolites_6y_for_ml.csv": "synthetic_metabolites_6y.csv",
}
if not all((DATA_DIR / new).exists() for new in _RENAMED.values()):
    import subprocess
    import shutil
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, str(SRC_DIR / "scripts" / "make_synthetic_data.py"),
                    "--outdir", str(DATA_DIR), "--force"], check=True, capture_output=True)
    subprocess.run([sys.executable, str(SRC_DIR / "scripts" / "build_inputs.py"),
                    "--source", str(DATA_DIR / "datacorr_6m6y18mMw1w24_final.csv"),
                    "--outdir", str(DATA_DIR)], check=True, capture_output=True)
    for old, new in _RENAMED.items():
        shutil.move(str(DATA_DIR / old), str(DATA_DIR / new))
    print("generated data/ (not shipped with the repo)")
# The synthetic data files are named distinctly from metabo.config's own default filenames,
# so every script that reads them -- in this process or in a subprocess started from here --
# is told explicitly which files to use.
os.environ["METABO_SOURCE"] = str(DATA_DIR / "synthetic_raw_metabolome.csv")
os.environ["METABO_TARGET_FILE"] = str(DATA_DIR / "synthetic_targets.csv")
os.environ["METABO_METAB_6M"] = str(DATA_DIR / "synthetic_metabolites_6m.csv")
os.environ["METABO_METAB_18M"] = str(DATA_DIR / "synthetic_metabolites_18m.csv")
os.environ["METABO_METAB_6Y"] = str(DATA_DIR / "synthetic_metabolites_6y.csv")
# The synthetic metabolites are named generically (metabolite001, ...), not with real
# chemical nomenclature, so the attribution scripts are told which generic name is the
# planted fish-oil marker (and its weaker parent compound).
os.environ["METABO_FISHOIL_MARKER"] = "metabolite001"
os.environ["METABO_FISHOIL_MARKER_PARENT"] = "metabolite002"
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("NUMBA_THREADING_LAYER", "workqueue")

# Reduced from the study's settings (1000 bootstraps/permutations, all five embedding
# dimensions, ten repeats) so every cell below finishes in well under a minute.
os.environ["METABO_N_BOOTSTRAP"] = "100"
os.environ["METABO_N_PERMUTATION"] = "200"
os.environ["METABO_EMBED_DIMS"] = "2 16"
os.environ["METABO_VAE_MAX_EPOCHS"] = "40"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from metabo import config, data_io, evaluation

print("embedding dims used here:", config.EMBED_DIMS, "(study default: 2 3 4 8 16)")


embedding dims used here: [2, 16] (study default: 2 3 4 8 16)


## 1. The below-detection fill rule

The main analysis fills values below detection with one tenth of the metabolite's minimum,
pooled over all three ages, once before any modeling (`scripts/build_inputs.py`). Two
alternatives (see `scripts/run_fill_sensitivity.py`'s docstring for all four arms):
`minfill` (fill = the minimum itself, no /10) and `perage` (minimum at the sample's own age,
divided by 10, instead of pooling ages).


In [2]:
raw_source = pd.read_csv(config.SOURCE_FILE, sep=";", decimal=",", low_memory=False)
raw_source = raw_source[raw_source["TPn"] == "6m"].copy()
metab_cols = list(pd.read_csv(config.METAB_FILES["6m"], index_col=0, nrows=0).columns)
met = raw_source[metab_cols].apply(pd.to_numeric, errors="coerce")


def fill_variant(arm):
    if arm == "baseline":
        logd = np.log10(met.fillna(met.min(axis=0) / 10.0))
    elif arm == "minfill":
        logd = np.log10(met.fillna(met.min(axis=0)))
    elif arm == "perage":
        # only one age (6m) is loaded here, so "per age" and "pooled" coincide; the real
        # script pools the three ages and differs there (see its docstring).
        logd = np.log10(met.fillna(met.min(axis=0) / 10.0))
    logd.index = raw_source["ABCno"].values
    logd.index.name = "id"
    return logd


targets = data_io.load_targets()
rows = []
for arm in ["baseline", "minfill", "perage"]:
    X = fill_variant(arm)
    X, y = data_io.align(X, targets["sex"])
    for rep, dim in [("raw", 2), ("pca", 16)]:
        rec, _ = evaluation.evaluate(X, y, rep, dim, "lr", random_state=config.SEED)
        rows.append({"arm": arm, "representation": rep, "auroc": round(rec["auroc"], 3)})

pd.DataFrame(rows).pivot(index="arm", columns="representation", values="auroc")


representation,pca,raw
arm,,
baseline,0.528,0.751
minfill,0.514,0.765
perage,0.528,0.751


(`baseline` should match the AUROC already seen for `sex`/raw and `sex`/pca in the
other notebooks, since it is the same fill rule, rebuilt here from the source file as a check.)

## 2. The scaler

`RobustScaler` (median and interquartile range) is the main analysis's choice, because a few
metabolites have extreme filled values. The alternative is `StandardScaler` (mean and standard
deviation). `evaluation.build_embed_pipeline` always asks for `RobustScaler` by name, so
`run_scaling_sensitivity.py`'s trick is to monkeypatch that name inside
`metabo.evaluation` itself; the same patch, applied and reverted here.


In [3]:
from sklearn.preprocessing import StandardScaler, RobustScaler

X_6m = data_io.load_metabolites("6m")
X, y = data_io.align(X_6m, targets["sex"])

rows = []
for rep, dim in [("raw", 2), ("pca", 16)]:
    rec, _ = evaluation.evaluate(X, y, rep, dim, "lr", random_state=config.SEED)
    rows.append({"scaler": "RobustScaler (baseline)", "representation": rep,
                 "auroc": round(rec["auroc"], 3)})

evaluation.RobustScaler = StandardScaler
try:
    for rep, dim in [("raw", 2), ("pca", 16)]:
        rec, _ = evaluation.evaluate(X, y, rep, dim, "lr", random_state=config.SEED)
        rows.append({"scaler": "StandardScaler", "representation": rep,
                     "auroc": round(rec["auroc"], 3)})
finally:
    evaluation.RobustScaler = RobustScaler  # revert the patch

pd.DataFrame(rows).pivot(index="scaler", columns="representation", values="auroc")


representation,pca,raw
scaler,,
RobustScaler (baseline),0.529,0.751
StandardScaler,0.537,0.743


## 3. Fill estimated on training folds only

In the main analysis the fill value is set once, on all children, before any cross-validation
split. Here it is re-estimated inside each training fold instead (held-out children excluded),
with a small transformer that is the first step of the embedding pipeline, following
`scripts/run_trainonly_sensitivity.py`'s own `TrainOnlyFill`.


In [4]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline


class TrainOnlyFill(BaseEstimator, TransformerMixin):
    '''Fill fitted on the training rows only; the first column of X is the child ID,
    kept only so fit() can see it and dropped again on transform().'''

    def fit(self, X, y=None):
        values = X[:, 1:]
        self.fill_ = np.nanmin(values, axis=0) - 1.0  # log10 scale: minimum / 10
        return self

    def transform(self, X):
        values = X[:, 1:].copy()
        mask = np.isnan(values)
        values[mask] = np.take(self.fill_, np.where(mask)[1])
        return values


# log10 values with missing values kept (not pre-filled), 6 months only
nan_logd = np.log10(met)
nan_logd.index = raw_source["ABCno"].values
nan_logd.index.name = "id"
Xf, y = data_io.align(nan_logd, targets["sex"])
Xid = Xf.copy()
Xid.insert(0, "__id__", Xid.index.values.astype(float))

_unpatched_build = evaluation.build_embed_pipeline


def patched_build(representation, dim, random_state, n_neighbors=None):
    base = _unpatched_build(representation, dim, random_state, n_neighbors)
    return Pipeline([("trainfill", TrainOnlyFill())] + base.steps)


evaluation.build_embed_pipeline = patched_build
try:
    rows = []
    for rep, dim in [("raw", 2), ("pca", 16)]:
        rec, _ = evaluation.evaluate(Xid, y, rep, dim, "lr", random_state=config.SEED)
        rows.append({"fill": "estimated per training fold", "representation": rep,
                     "auroc": round(rec["auroc"], 3)})
finally:
    evaluation.build_embed_pipeline = _unpatched_build  # revert the patch

pd.DataFrame(rows)


,fill,representation,auroc
0,estimated per training fold,raw,0.742
1,estimated per training fold,pca,0.529


Compare this against the `baseline` row of the fill-rule table above (same fill rule
in spirit, estimated once upfront there vs. separately per fold here); a large difference would
suggest the main analysis's upfront fill was leaking information about the held-out rows.

## Next steps

- Run the full grid (all three sensitivity analyses, all three positive-control timepoints,
  every representation/dimension/classifier) as batch jobs:

```bash
cd ../src
export METABO_DATA_DIR=../data METABO_RESULTS_DIR=../notebooks/results
python scripts/run_fill_sensitivity.py --dims 2 16
python scripts/run_scaling_sensitivity.py
python scripts/run_trainonly_sensitivity.py --arm fill --one-pair
```

